# 01 — Diagnóstico dos dados e correção da planilha de variáveis

**Objetivo:** inventariar os 20 shapefiles de origem (5 cidades × [2000 urbano, 2000 rural, 2010, 2022]),
validar que as colunas correspondem à numeração do dicionário de variáveis, e produzir uma cópia corrigida da
planilha de fórmulas — fonte de verdade para o cálculo das variáveis derivadas no notebook 05.

**Entradas:**
- `input/<Cidade>/<ano>/*.shp` (20 arquivos)
- `input/VARIÁVEIS_censos 2000_2010_2022.xlsx`

**Saídas:**
- `output/01_diagnostico/totais_baseline_por_cidade_ano.csv`
- `output/01_diagnostico/VARIAVEIS_censos_2000_2010_2022_CORRIGIDA.xlsx`

**Depende de:** nenhum (primeiro notebook do pipeline).

## 1. Setup

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "utils.py").exists())
sys.path.insert(0, str(REPO_ROOT / "src"))

import geopandas as gpd
import pandas as pd
import openpyxl

from utils import (
    CIDADES, ANOS, INPUT_DIR, OUTPUT_DIR,
    caminho_shapefile, ENCODING_POR_ANO,
)

OUT_DIR = OUTPUT_DIR / "01_diagnostico"
OUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)


## 2. Inventário dos 20 shapefiles

- Para cada cidade e ano, lê a geometria (via `pyogrio`, sem carregar a malha inteira em memória mais de uma
  vez) e registra: nº de feições, CRS declarado, colunas presentes, e extensão (bounding box).
- Usado no notebook 02 para diagnosticar o problema de projeção de 2000.

In [ ]:
import pyogrio

registros = []

for cidade in CIDADES:
    tarefas = [(2000, "urbano"), (2000, "rural"), (2010, None), (2022, None)]
    for ano, tipo in tarefas:
        caminho = caminho_shapefile(cidade, ano, tipo)
        info = pyogrio.read_info(caminho)
        registros.append({
            "cidade": cidade,
            "ano": ano,
            "tipo": tipo or "-",
            "arquivo": caminho.name,
            "n_feicoes": info["features"],
            "crs": info["crs"],
            "colunas": ", ".join(info["fields"]),
            "xmin": info["total_bounds"][0],
            "ymin": info["total_bounds"][1],
            "xmax": info["total_bounds"][2],
            "ymax": info["total_bounds"][3],
        })

inventario = pd.DataFrame(registros)
inventario


Observação esperada: os `.prj` de 2000-urbano declaram zonas UTM **Norte** (`EPSG:326XX`), mas os municípios estão
no hemisfério Sul. Isso fica evidente comparando `ymin`/`ymax` — coordenadas UTM Sul têm northing de ~9.000.000 a
~10.000.000 m (por causa do false northing de 10.000.000), enquanto zonas Norte usam false northing 0. A correção
efetiva é feita no notebook 02; aqui só deixamos o diagnóstico registrado.

In [ ]:
inventario[["cidade", "ano", "tipo", "crs", "ymin", "ymax"]]


## 3. Dicionário de variáveis: leitura e validação

- Carrega as duas abas da planilha de dicionário de variáveis.
- Da aba `VARIÁVEIS FINAL`, extrai variável, tema, censos em que existe, fórmula de cálculo (quando aplicável)
  e observações.

In [ ]:
XLSX_ORIGINAL = INPUT_DIR / "VARIÁVEIS_censos 2000_2010_2022.xlsx"

origem = pd.read_excel(XLSX_ORIGINAL, sheet_name="VARIÁVEIS DE BAS 2000_2010_2022", header=1)
final = pd.read_excel(XLSX_ORIGINAL, sheet_name="VARIÁVEIS FINAL", header=3)
final = final.dropna(subset=["VARIÁVEL"]).reset_index(drop=True)

final


## 4. Validação das identidades de fechamento

- Testa, feição a feição, se a soma dos domicílios por espécie (casa + casa de vila/condomínio + apartamento)
  fecha com o total de domicílios particulares permanentes/ocupados, e o mesmo para moradores.
- Confirma que os *nomes* de coluna dos shapefiles já correspondem à numeração do dicionário (sem deslocamento),
  e detecta feições com erro de preenchimento (ex.: join mal feito na fonte).

In [ ]:
def carrega_atributos(cidade, ano, tipo=None):
    caminho = caminho_shapefile(cidade, ano, tipo)
    gdf = gpd.read_file(caminho, encoding=ENCODING_POR_ANO[ano])
    gdf["_cidade"] = cidade
    gdf["_ano"] = ano
    gdf["_tipo"] = tipo or "-"
    return gdf


def valida_fechamento(gdf, cidade, ano, tipo):
    '''Confere V28+V32+V36=V26 (domicílios por espécie) e V30+V34+V38=V27 (moradores por espécie).'''
    problemas = []
    cols = gdf.columns
    if {"V26", "V28", "V32", "V36"}.issubset(cols):
        soma_dom = gdf[["V28", "V32", "V36"]].fillna(0).sum(axis=1)
        diff_dom = (soma_dom - gdf["V26"]).abs()
        n_falhas = int((diff_dom > 0.5).sum())
        problemas.append({"cidade": cidade, "ano": ano, "tipo": tipo or "-",
                           "checagem": "V28+V32+V36 == V26", "n_feicoes": len(gdf),
                           "n_falhas": n_falhas, "max_diff": float(diff_dom.max())})
    if {"V27", "V30", "V34", "V38"}.issubset(cols):
        soma_mor = gdf[["V30", "V34", "V38"]].fillna(0).sum(axis=1)
        diff_mor = (soma_mor - gdf["V27"]).abs()
        n_falhas = int((diff_mor > 0.5).sum())
        problemas.append({"cidade": cidade, "ano": ano, "tipo": tipo or "-",
                           "checagem": "V30+V34+V38 == V27", "n_feicoes": len(gdf),
                           "n_falhas": n_falhas, "max_diff": float(diff_mor.max())})
    if {"V21", "V22"}.issubset(cols):
        n_falhas = int((gdf["V22"] > gdf["V21"]).sum())
        problemas.append({"cidade": cidade, "ano": ano, "tipo": tipo or "-",
                           "checagem": "V22 <= V21", "n_feicoes": len(gdf),
                           "n_falhas": n_falhas, "max_diff": None})
    return problemas


relatorio_fechamento = []
gdfs_cache = {}
for cidade in CIDADES:
    for ano, tipo in [(2000, "urbano"), (2000, "rural"), (2010, None), (2022, None)]:
        gdf = carrega_atributos(cidade, ano, tipo)
        gdfs_cache[(cidade, ano, tipo)] = gdf
        relatorio_fechamento.extend(valida_fechamento(gdf, cidade, ano, tipo))

df_fechamento = pd.DataFrame(relatorio_fechamento)
df_fechamento


Se `n_falhas` for 0 (ou próximo de 0) em todas as linhas, confirma-se que os nomes de coluna dos shapefiles já
correspondem à numeração do dicionário — não há necessidade de rename. Falhas isoladas indicam erro pontual de
preenchimento na fonte (ex.: join incompleto), não um problema estrutural de nomenclatura.

In [ ]:
pior_caso = df_fechamento.sort_values("n_falhas", ascending=False).head(10)
pior_caso


## 4.1. Colunas inteiramente nulas onde deveriam ter dado

- Checagem coluna a coluna: variáveis que deveriam existir naquele ano (diferente do achado "V9-V19 não existe
  em 2000"), mas vieram quase 100% nulas por inconsistência no arquivo de origem.
- Diferente de supressão estatística pontual do IBGE, que afeta poucas feições.

In [ ]:
from utils import VARS_AUSENTES_2000

VARS_TODAS = ["V1", "V3", "V5", "V7", "V9", "V11", "V13", "V15", "V17", "V19", "V21", "V22",
              "V24", "V25", "V26", "V27", "V28", "V30", "V32", "V34", "V36", "V38",
              "V40", "V42", "V44", "V46", "V48", "V50", "V52", "V54"]

relatorio_nulos = []
for (cidade, ano, tipo), gdf in gdfs_cache.items():
    for var in VARS_TODAS:
        if var not in gdf.columns:
            continue
        pct_nulo = 100 * gdf[var].isna().mean()
        esperado_ausente = (ano == 2000 and var in VARS_AUSENTES_2000)
        if pct_nulo > 90 and not esperado_ausente:
            relatorio_nulos.append({"cidade": cidade, "ano": ano, "tipo": tipo, "variavel": var,
                                     "pct_nulo": round(pct_nulo, 1), "n_feicoes": len(gdf)})

df_nulos_suspeitos = pd.DataFrame(relatorio_nulos)
df_nulos_suspeitos


Cada linha aqui é uma inconsistência do arquivo de origem a registrar: a variável deveria existir naquele ano,
mas está inteiramente (ou quase) nula.

## 5. Totais municipais de referência (baseline)

- Totais de V1 (população), V26 (domicílios) e V27 (DPP/ocupado) por cidade e ano, na malha de origem.
- Usado no notebook 03 para validar que a interpolação areal preserva os totais depois da normalização.

In [ ]:
linhas_totais = []
for cidade in CIDADES:
    for ano in ANOS:
        if ano == 2000:
            partes = [gdfs_cache[(cidade, 2000, "urbano")], gdfs_cache[(cidade, 2000, "rural")]]
            gdf = pd.concat(partes, ignore_index=True)
        else:
            gdf = gdfs_cache[(cidade, ano, None)]
        linha = {"cidade": cidade, "ano": ano, "n_setores": len(gdf)}
        for v in ["V1", "V26", "V27"]:
            linha[v] = gdf[v].sum() if v in gdf.columns else None
        linhas_totais.append(linha)

totais_baseline = pd.DataFrame(linhas_totais)
totais_baseline.to_csv(OUT_DIR / "totais_baseline_por_cidade_ano.csv", index=False, encoding="utf-8-sig")
totais_baseline


## 6. Correção das fórmulas de percentual (coluna D da planilha)

- Bloco de população (V4 a V23): fórmulas corretas — numerador é a contagem da linha imediatamente acima,
  denominador é V1 (ou V21 para alfabetização).
- Bloco de domicílios (V29 a V55): a coluna D referencia a própria variável no numerador — impossível
  (`V29 = V29 × 100 / V27`).
- Regra de correção, validada contra os rótulos da coluna B (`TEMA`): numerador é a contagem da linha
  imediatamente acima (`V(n-1)`); denominador é **V26** para percentuais de domicílios ou **V27** para
  percentuais de moradores (identificado pela palavra "morador" no texto do tema).

In [ ]:
import re

VAR_RE = re.compile(r"V\d+A?")


def variaveis_citadas(formula):
    '''Extrai as variáveis V* citadas numa fórmula, tolerando o typo "V 47" (espaço) achado na planilha de
    origem — sem essa normalização, a auto-referência dessa linha não seria detectada.'''
    if not isinstance(formula, str):
        return []
    formula_normalizada = re.sub(r"V\s+(\d)", r"V\1", formula)
    return VAR_RE.findall(formula_normalizada)


def corrige_formula(row):
    '''Retorna (formula_corrigida, foi_alterada) para uma linha da aba VARIÁVEIS FINAL.'''
    var = row["VARIÁVEL"]
    formula = row["FÓRMULA / DENOMINADOR"]
    if not isinstance(formula, str) or not formula.strip():
        return "", False

    citadas = variaveis_citadas(formula)
    if var not in citadas:
        # Fórmula já é consistente (bloco de população) — mantém como está.
        return formula.strip(), False

    # Auto-referencial: numerador = contagem imediatamente acima; denominador por tema.
    numero = int(re.sub(r"\D", "", var))
    numerador = f"V{numero - 1}"
    tema = str(row["TEMA"]) if pd.notna(row["TEMA"]) else ""
    denominador = "V27" if "morador" in tema.lower() else "V26"
    corrigida = f"{numerador} X 100 / {denominador}"
    return corrigida, True


resultado = final.apply(corrige_formula, axis=1, result_type="expand")
resultado.columns = ["FORMULA_CORRIGIDA", "ALTERADA"]
final_corrigido = pd.concat([final, resultado], axis=1)

final_corrigido[final_corrigido["ALTERADA"]][
    ["VARIÁVEL", "TEMA", "FÓRMULA / DENOMINADOR", "FORMULA_CORRIGIDA"]
]


## 7. Geração da planilha corrigida

- Copia o XLSX original e insere, na aba `VARIÁVEIS FINAL`, duas colunas à direita da coluna D (`FÓRMULA /
  DENOMINADOR`): a fórmula corrigida e um marcador de alteração.
- A coluna `OBSERVAÇÃO` original é preservada, apenas deslocada.
- Premissa: o arquivo original em `input/` não é modificado; a cópia corrigida sai em `output/`.

In [ ]:
from openpyxl.styles import PatternFill

XLSX_CORRIGIDO = OUT_DIR / "VARIAVEIS_censos_2000_2010_2022_CORRIGIDA.xlsx"

wb = openpyxl.load_workbook(XLSX_ORIGINAL)
ws = wb["VARIÁVEIS FINAL"]

HEADER_ROW = 4  # linha 4 = cabeçalho (VARIÁVEL, TEMA, CENSO, FÓRMULA / DENOMINADOR, OBSERVAÇÃO)
COL_FORMULA_ORIGINAL = 4  # D

# Insere 2 colunas novas à direita de D, empurrando a antiga E (OBSERVAÇÃO) para G.
ws.insert_cols(COL_FORMULA_ORIGINAL + 1, amount=2)
ws.cell(row=HEADER_ROW, column=COL_FORMULA_ORIGINAL + 1, value="FÓRMULA / DENOMINADOR (corrigida)")
ws.cell(row=HEADER_ROW, column=COL_FORMULA_ORIGINAL + 2, value="ALTERADA?")

destaque = PatternFill(start_color="FFF59D", end_color="FFF59D", fill_type="solid")

# Mapa VARIÁVEL -> (formula_corrigida, alterada), na ordem de leitura do pandas.
mapa_correcao = dict(zip(final_corrigido["VARIÁVEL"], zip(final_corrigido["FORMULA_CORRIGIDA"], final_corrigido["ALTERADA"])))

for row_idx in range(HEADER_ROW + 1, ws.max_row + 1):
    var = ws.cell(row=row_idx, column=1).value
    if var not in mapa_correcao:
        continue
    formula_corrigida, alterada = mapa_correcao[var]
    cel_formula = ws.cell(row=row_idx, column=COL_FORMULA_ORIGINAL + 1, value=formula_corrigida)
    cel_flag = ws.cell(row=row_idx, column=COL_FORMULA_ORIGINAL + 2, value="SIM" if alterada else "não")
    if alterada:
        cel_formula.fill = destaque
        cel_flag.fill = destaque
        ws.cell(row=row_idx, column=COL_FORMULA_ORIGINAL).fill = destaque

wb.save(XLSX_CORRIGIDO)
print(f"Planilha corrigida salva em: {XLSX_CORRIGIDO}")
print(f"Linhas alteradas: {int(final_corrigido['ALTERADA'].sum())} de {len(final_corrigido)}")
